In [1]:
import os
import pandas as pd

In [ ]:
## Stack data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso into one file for hourly day ahead prices 
## Same code also works for load forecast data, and renewable generation data

# Define the base directory for Day ahead price data
base_path = "Data/unproc/"

# Define available years
years = [2020, 2021, 2022, 2023, 2024]

# Initialize a list to store dataframes
data_list = []

# Loop through each year and process data
for year in years:
    file_path = os.path.join(base_path, f"caiso_lmp_da_hr_zones_{year}.csv")

    if os.path.exists(file_path):  # Check if file exists before loading
        df = pd.read_csv(file_path, skiprows=3, delimiter=",")
        df["year"] = year  # Add a year column
        data_list.append(df)
        print(f"✅ Loaded: {file_path}")
    else:
        print(f"❌ Missing file: {file_path}")

# Concatenate all available data
if data_list:
    stacked_data = pd.concat(data_list, ignore_index=True)

    # Save the stacked dataset
    output_file_path = "Data/preprocessed/stacked_lmp_da_hr_data.csv"
    stacked_data.to_csv(output_file_path, index=False)
    print(f"✅ Stacked data saved to: {output_file_path}")

else:
    print("❌ No files were found. Please check the directory.")


In [ ]:
## Stack real time price data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso into one file


# Define the base directory
base_path = "Data/unproc/"

# Define the range of years and quarters
years = range(2020, 2026)  # Covers 2020 to 2025
quarters = ["Q1", "Q2", "Q3", "Q4"]

# Initialize a list to store dataframes
price_data_list = []

# Loop through each year and quarter to load and append preprocessed LMP data
for year in years:
    for quarter in quarters:
        quarter_str = f"{year}{quarter}"
        if quarter_str < "2020Q4" or quarter_str > "2025Q1":  # Skip out-of-range quarters
            continue

        # Construct the correct file path
        file_path = f"/content/drive/MyDrive/PowerPrices/Data/preprocessed/{year}/preprocessed_lmp_rt_{year}{quarter}.csv"

        # Load the file
        df = pd.read_csv(file_path)
        df["quarter"] = quarter_str  # Add a column to identify the quarter
        price_data_list.append(df)
        print(f"✅ Loaded: {file_path}")

# Concatenate all LMP price dataframes into a single dataframe
stacked_lmp_data = pd.concat(price_data_list, ignore_index=True)

# Define the output path for the stacked LMP data
output_file_path = "/content/drive/MyDrive/PowerPrices/Data/preprocessed/stacked_lmp_price_data.csv"

# Save the stacked dataframe as a CSV file
stacked_lmp_data.to_csv(output_file_path, index=False)
print(f"✅ Stacked LMP price data saved to: {output_file_path}")

# # Display the stacked dataframe
# import ace_tools as tools
# tools.display_dataframe_to_user(name="Stacked LMP Price Data", dataframe=stacked_lmp_data)


In [5]:
# Remove duplicate times by replacing with mean value and insert missing times

def process_begin_times(df):
    df['begin_time'] = pd.to_datetime(df['begin_time'])
    df.set_index('begin_time', inplace=True)
    df = df.groupby(df.index).mean()
    dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1h')
    df = df.reindex(dates)
    df.reset_index(inplace=True)
    df.rename(columns={'index':'begin_time'}, inplace=True)
    return df 

The following code prepares exogenous variables for price forecasting.

In [ ]:
## Hourly load data downloaded from gridstatus.io

load_data = pd.read_csv('Data/load_1hr_PGE.csv', usecols=['interval_start_local','load'])
load_data.rename(columns={'interval_start_local':'begin_time'}, inplace=True)
load_data['begin_time'] = pd.to_datetime(load_data['begin_time']).apply(lambda x : x.tz_localize(None))
load_data = process_begin_times(load_data)
load_data

/var/folders/y1/qbgxvx6j1vj5ydy42yxx7fsm0000gn/T/ipykernel_59241/3809692419.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  load_data['begin_time'] = pd.to_datetime(load_data['begin_time']).apply(lambda x : x.tz_localize(None))


,begin_time,load
0,2020-01-01 00:00:00,10206.0
1,2020-01-01 01:00:00,9866.0
2,2020-01-01 02:00:00,9605.0
3,2020-01-01 03:00:00,9453.0
4,2020-01-01 04:00:00,9389.0
...,...,...
43843,2024-12-31 19:00:00,11307.0
43844,2024-12-31 20:00:00,10967.0
43845,2024-12-31 21:00:00,10614.0
43846,2024-12-31 22:00:00,10226.0


In [ ]:
load_forecast = pd.read_csv('Data/preprocessed/stacked_load_forecast_data.csv', usecols=['Local Timestamp Pacific Time (Interval Beginning)', 'Pacific Gas and Electric Forecast Load (MW)'])
load_forecast.rename(columns={'Local Timestamp Pacific Time (Interval Beginning)':'begin_time'}, inplace=True)
load_forecast=process_begin_times(load_forecast)
load_forecast
load_all = pd.merge(load_data, load_forecast, on='begin_time', how='inner')
load_all

,begin_time,load,Pacific Gas and Electric Forecast Load (MW)
0,2021-03-12 00:00:00,9914.0,10102.72
1,2021-03-12 01:00:00,9827.0,9868.57
2,2021-03-12 02:00:00,9849.0,9712.50
3,2021-03-12 03:00:00,9939.0,9715.76
4,2021-03-12 04:00:00,10287.0,9969.51
...,...,...,...
33379,2024-12-31 19:00:00,11307.0,11959.79
33380,2024-12-31 20:00:00,10967.0,11723.02
33381,2024-12-31 21:00:00,10614.0,11408.80
33382,2024-12-31 22:00:00,10226.0,10831.46


In [ ]:
## Hourly solar/wind generaction forecast data downloaded from gridstatus.io


renforecast = pd.read_csv('Data/solarwind_forecast_1hr_NP15.csv', usecols=['interval_start_local', 'solar_mw', 'wind_mw'])
renforecast.rename(columns={'interval_start_local':'begin_time'}, inplace=True)
renforecast = renforecast.groupby('begin_time').last() #Pick the most recent forecast when multiple forecasts for the same time are available
renforecast.reset_index(inplace=True)
renforecast.rename(columns={'index':'begin_time', 'solar_mw':'solar_forecast', 'wind_mw':'wind_forecast'}, inplace=True)
renforecast['begin_time'] = pd.to_datetime(renforecast['begin_time']).apply(lambda x : x.tz_localize(None))
renforecast = process_begin_times(renforecast)
renforecast

/var/folders/y1/qbgxvx6j1vj5ydy42yxx7fsm0000gn/T/ipykernel_59241/1507475549.py:6: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  renforecast['begin_time'] = pd.to_datetime(renforecast['begin_time']).apply(lambda x : x.tz_localize(None))


,begin_time,solar_forecast,wind_forecast
0,2021-01-01 00:00:00,0.00,99.27
1,2021-01-01 01:00:00,0.00,68.52
2,2021-01-01 02:00:00,0.00,59.47
3,2021-01-01 03:00:00,0.00,59.21
4,2021-01-01 04:00:00,0.00,48.41
...,...,...,...
37027,2025-03-23 19:00:00,4.86,100.60
37028,2025-03-23 20:00:00,0.00,144.92
37029,2025-03-23 21:00:00,0.00,170.87
37030,2025-03-23 22:00:00,0.00,184.65


In [28]:
ren_gen_actual = pd.read_csv('Data/preprocessed/stacked_gen_ren_hr_data.csv', usecols=['Local Timestamp Pacific Time (Interval Beginning)','NP15 Solar Generation (MW)','NP15 Wind Generation (MW)'])
ren_gen_actual.rename(columns={'Local Timestamp Pacific Time (Interval Beginning)':'begin_time'}, inplace=True)
ren_gen_actual = process_begin_times(ren_gen_actual)
ren_all = pd.merge(renforecast, ren_gen_actual, on='begin_time', how='inner')
ren_all

,begin_time,solar_forecast,wind_forecast,NP15 Solar Generation (MW),NP15 Wind Generation (MW)
0,2021-01-01 00:00:00,0.0,99.27,-3.88458,179.27404
1,2021-01-01 01:00:00,0.0,68.52,-3.91677,97.43401
2,2021-01-01 02:00:00,0.0,59.47,-3.94931,41.32052
3,2021-01-01 03:00:00,0.0,59.21,-3.95021,20.37190
4,2021-01-01 04:00:00,0.0,48.41,-3.96713,43.17521
...,...,...,...,...,...
35059,2024-12-31 19:00:00,0.0,48.25,-3.03842,58.35641
35060,2024-12-31 20:00:00,0.0,55.64,-7.19792,65.56839
35061,2024-12-31 21:00:00,0.0,65.90,-6.79170,62.77812
35062,2024-12-31 22:00:00,0.0,77.46,-6.26466,54.79276


In [ ]:
## Henry Hub natural gas spot prices downloaded from https://www.eia.gov/dnav/ng/hist/rngwhhdm.htm
## This code puts prices in a series with time stamps at one hour intervals

natural_gas_price = pd.read_csv('Data/Henry_Hub_Natural_Gas_Spot_Price.csv', skiprows=4)
natural_gas_price.rename(columns={'Henry Hub Natural Gas Spot Price Dollars per Million Btu':'Natural_gas_price'}, inplace=True)
natural_gas_price['Day'] = pd.to_datetime(natural_gas_price['Day'])
natural_gas_price = natural_gas_price[::-1]
natural_gas_price.set_index('Day', inplace=True)
dates = pd.date_range(start=natural_gas_price.index.min(), end=natural_gas_price.index.max(), freq='1h')
natural_gas_price = natural_gas_price.reindex(dates)
natural_gas_price['Natural_gas_price'] = natural_gas_price['Natural_gas_price'].ffill()
natural_gas_price.reset_index(inplace=True)
natural_gas_price.rename(columns={'index':'begin_time'}, inplace=True)
natural_gas_price.head(50)
natural_gas_price.to_csv('Data/preprocessed/nat_gas_price.csv', index=False)

In [ ]:
## Combine to form one exogenous variables dataframe

load_gen = pd.merge(load_all, ren_all, on='begin_time', how='inner')
nat_gas_price = pd.read_csv('Data/preprocessed/nat_gas_price.csv')
nat_gas_price['begin_time'] = pd.to_datetime(nat_gas_price['begin_time'])
exog = pd.merge(load_gen, nat_gas_price[['begin_time', 'Natural_gas_price']], how='inner', on='begin_time')
exog.to_csv('Data/preprocessed/NP15_exog.csv', index=False)
exog

,begin_time,load,Pacific Gas and Electric Forecast Load (MW),solar_forecast,wind_forecast,NP15 Solar Generation (MW),NP15 Wind Generation (MW),Natural_gas_price
0,2021-03-12 00:00:00,9914.0,10102.72,0.0,327.97,-2.69832,526.78168,2.65
1,2021-03-12 01:00:00,9827.0,9868.57,0.0,259.94,-2.66009,405.97186,2.65
2,2021-03-12 02:00:00,9849.0,9712.50,0.0,191.44,-2.68074,288.43194,2.65
3,2021-03-12 03:00:00,9939.0,9715.76,0.0,148.07,-2.68692,205.85381,2.65
4,2021-03-12 04:00:00,10287.0,9969.51,0.0,134.36,-2.58009,173.78675,2.65
...,...,...,...,...,...,...,...,...
33379,2024-12-31 19:00:00,11307.0,11959.79,0.0,48.25,-3.03842,58.35641,3.40
33380,2024-12-31 20:00:00,10967.0,11723.02,0.0,55.64,-7.19792,65.56839,3.40
33381,2024-12-31 21:00:00,10614.0,11408.80,0.0,65.90,-6.79170,62.77812,3.40
33382,2024-12-31 22:00:00,10226.0,10831.46,0.0,77.46,-6.26466,54.79276,3.40
